<img src="https://www.fciencias.unam.mx/sites/default/files/logoFC_2.png" width="500" align="left"/>
<p align="right">
<b>Lingüística Computacional</b>
<br><b>Práctica 4: Evaluación de modelos del lenguaje neuronales</b>
<br><b>Profesora:</b> Dra. Ximena Gutiérrez Vasques
<br><b>Ayudante Teo:</b> Ximena de la Luz Contreras Mendoza
<br><b>Ayudante Lab:</b> Diego Alberto Barriga Martínez
<br><b>Alumna:</b> Ortega Hernández Zaira Daniela
<br><b>Mayo, 2026</b>
</p>

##**Práctica 4: Evaluación de modelos del lenguaje neuronales**


In [ ]:
!pip install torch transformers nltk numpy datasets gensim sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.4 MB/s eta 0:00:00


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from nltk.corpus import genesis, gutenberg, inaugural, state_union, webtext, abc
from nltk import ngrams
from collections import Counter
from transformers import AutoTokenizer
import nltk

In [ ]:
# Descargar recursos necesarios
nltk.download('genesis')
nltk.download('gutenberg')
nltk.download('inaugural')
nltk.download('state_union')
nltk.download('webtext')
nltk.download('abc')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package genesis to /root/nltk_data...
[nltk_data]   Unzipping corpora/genesis.zip.
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package inaugural to /root/nltk_data...
[nltk_data]   Unzipping corpora/inaugural.zip.
[nltk_data] Downloading package state_union to /root/nltk_data...
[nltk_data]   Unzipping corpora/state_union.zip.
[nltk_data] Downloading package webtext to /root/nltk_data...
[nltk_data]   Unzipping corpora/webtext.zip.
[nltk_data] Downloading package abc to /root/nltk_data...
[nltk_data]   Unzipping corpora/abc.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Configuración
UNK_LABEL = "<UNK>"
BOS_LABEL = "<BOS>"
EOS_LABEL = "<EOS>"
EMBEDDING_DIM = 200
CONTEXT_SIZE = 2
BATCH_SIZE = 256
H = 100
EPOCHS = 3

In [ ]:

# ==================== FUNCIONES AUXILIARES ====================

def preprocess_corpus_with_subword(corpus, tokenizer):
    """Tokeniza corpus usando subword tokenization y añade marcadores especiales"""
    processed = []
    for sent in corpus:
        # Convertir sent (tuple) a string
        sent_str = ' '.join(sent)
        # Tokenizar con subword
        subwords = tokenizer.tokenize(sent_str)
        # Añadir marcadores especiales
        subwords = [BOS_LABEL] + subwords + [EOS_LABEL]
        processed.append(subwords)
    return processed

def preprocess_corpus_word_based(corpus):
    """Preprocesa corpus con tokenización por palabras"""
    processed = []
    for sent in corpus:
        # Convertir sent a lista y añadir marcadores
        words = [word.lower() for word in sent]
        words = [BOS_LABEL] + words + [EOS_LABEL]
        processed.append(words)
    return processed

def build_vocabulary(corpus, min_freq=1):
    """Construye vocabulario a partir del corpus"""
    word_freqs = Counter()
    for sentence in corpus:
        for word in sentence:
            word_freqs[word] += 1

    # Filtrar palabras poco frecuentes
    vocab = {UNK_LABEL: 0, BOS_LABEL: 1, EOS_LABEL: 2}
    idx = 3

    for word, freq in word_freqs.items():
        if freq >= min_freq and word not in vocab:
            vocab[word] = idx
            idx += 1

    return vocab, {v: k for k, v in vocab.items()}

def get_word_id(vocab, word):
    """Obtiene ID de palabra, usando UNK si no existe"""
    return vocab.get(word, vocab[UNK_LABEL])

def create_ngram_data(corpus, vocab, n=3):
    """Crea datos de entrenamiento a partir de n-gramas"""
    x_data = []
    y_data = []

    for sentence in corpus:
        # Verificar que la oración tenga suficientes tokens
        if len(sentence) < n:
            continue

        n_grams = ngrams(sentence, n)
        for gram in n_grams:
            # Los primeros n-1 tokens son el contexto
            context = [get_word_id(vocab, w) for w in gram[:-1]]
            # El último token es el target
            target = get_word_id(vocab, gram[-1])
            x_data.append(context)
            y_data.append(target)

    return np.array(x_data), np.array(y_data)

In [ ]:
# ==================== MODELO ====================

class TrigramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size, hidden_dim):
        super(TrigramModel, self).__init__()
        self.context_size = context_size
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(0.2)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view(-1, self.context_size * self.embeddings.embedding_dim)
        out = torch.tanh(self.linear1(embeds))
        out = self.dropout(out)
        out = self.linear2(out)
        log_probs = F.log_softmax(out, dim=1)
        return log_probs

def train_model(model, train_loader, vocab_size, device, epochs=EPOCHS):
    """Entrena el modelo"""
    loss_function = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

    for epoch in range(epochs):
        total_loss = 0
        model.train()

        for context, target in train_loader:
            context = context.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            log_probs = model(context)
            loss = loss_function(log_probs, target)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    return model

def calculate_perplexity(model, data_loader, device):
    """Calcula la perplexidad del modelo"""
    model.eval()
    total_loss = 0
    total_tokens = 0
    loss_function = nn.NLLLoss(reduction='sum')

    with torch.no_grad():
        for context, target in data_loader:
            context = context.to(device)
            target = target.to(device)

            log_probs = model(context)
            loss = loss_function(log_probs, target)

            total_loss += loss.item()
            total_tokens += target.size(0)

    # Perplexidad = exp(average_loss)
    avg_loss = total_loss / total_tokens
    perplexity = np.exp(avg_loss)

    return perplexity


In [ ]:
# ==================== GENERACIÓN DE TEXTO ====================

def generate_text(model, vocab, idx_to_word, tokenizer=None, max_tokens=30, temperature=0.8):
    """Genera texto usando el modelo entrenado"""
    device = next(model.parameters()).device

    # Contexto inicial
    context = [vocab[BOS_LABEL]] * CONTEXT_SIZE
    generated = []

    model.eval()

    for _ in range(max_tokens):
        context_tensor = torch.tensor([context[-CONTEXT_SIZE:]]).to(device)

        with torch.no_grad():
            log_probs = model(context_tensor)
            probs = torch.exp(log_probs / temperature).squeeze()

            # Sampling
            next_token = torch.multinomial(probs, 1).item()

        generated.append(next_token)
        context.append(next_token)

        # Parar si generamos EOS
        if idx_to_word.get(next_token) == EOS_LABEL:
            break

    # Decodificar dependiendo del tipo de tokenización
    if tokenizer:
        # Para subword: unir tokens y decodificar
        tokens = [idx_to_word.get(t, UNK_LABEL) for t in generated if idx_to_word.get(t) not in [BOS_LABEL, EOS_LABEL]]
        text = ''.join(tokens).replace('##', '')
    else:
        # Para word-based
        text = ' '.join([idx_to_word.get(t, UNK_LABEL) for t in generated])

    return text

In [ ]:
# ==================== FUNCIÓN PRINCIPAL ====================

def main():
    # Configurar dispositivo
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Usando dispositivo: {device}")

    # Cargar corpus
    print("\n=== CARGANDO CORPORA ===")

    # Corpus de entrenamiento (sin genesis)
    train_corpora = []
    corpus_names = {
        'gutenberg': gutenberg,
        'inaugural': inaugural,
        'state_union': state_union,
        'webtext': webtext,
        'abc': abc
    }

    for name, corpus in corpus_names.items():
        try:
            sents = list(corpus.sents())  # Convertir a lista
            train_corpora.extend(sents)
            print(f"  - {name}: {len(sents)} oraciones")
        except Exception as e:
            print(f"  - Error cargando {name}: {e}")

    # Corpus de prueba (genesis)
    test_sents = genesis.sents()
    print(f"  - genesis (test): {len(test_sents)} oraciones")

    # ==================== MODELO BASE (WORD-BASED) ====================
    print("\n=== ENTRENANDO MODELO BASE (Word-Based) ===")

    # Preprocesar
    train_word = preprocess_corpus_word_based(train_corpora)
    test_word = preprocess_corpus_word_based(test_sents)

    # Construir vocabulario
    vocab_word, idx_to_word_word = build_vocabulary(train_word, min_freq=3)
    print(f"Tamaño vocabulario word-based: {len(vocab_word)}")

    # Crear datos
    X_train_word, y_train_word = create_ngram_data(train_word, vocab_word)
    X_test_word, y_test_word = create_ngram_data(test_word, vocab_word)

    # Crear DataLoaders
    train_dataset_word = list(zip(X_train_word, y_train_word))
    test_dataset_word = list(zip(X_test_word, y_test_word))

    train_loader_word = DataLoader(train_dataset_word, batch_size=BATCH_SIZE, shuffle=True)
    test_loader_word = DataLoader(test_dataset_word, batch_size=BATCH_SIZE)

    # Entrenar modelo base
    model_word = TrigramModel(len(vocab_word), EMBEDDING_DIM, CONTEXT_SIZE, H).to(device)
    model_word = train_model(model_word, train_loader_word, len(vocab_word), device)

    # Evaluar perplexidad
    perplexity_word = calculate_perplexity(model_word, test_loader_word, device)
    print(f"Perplexidad modelo word-based: {perplexity_word:.2f}")

    # Calcular OOV rate
    all_test_words = set()
    for sent in test_word:
        all_test_words.update(sent)
    vocab_words_set = set(vocab_word.keys())
    oov_words = all_test_words - vocab_words_set
    oov_rate_word = len(oov_words) / len(all_test_words) if all_test_words else 0
    print(f"OOV Rate word-based: {oov_rate_word:.2%}")

    # ==================== MODELO SUBWORD ====================
    print("\n=== ENTRENANDO MODELO SUBWORD ===")

    # Usar tokenizador pre-entrenado (BPE de GPT-2)
    subword_tokenizer = AutoTokenizer.from_pretrained("gpt2")
    subword_tokenizer.add_special_tokens({
        'bos_token': BOS_LABEL,
        'eos_token': EOS_LABEL,
        'unk_token': UNK_LABEL
    })

    # Preprocesar con subword
    train_subword = preprocess_corpus_with_subword(train_corpora, subword_tokenizer)
    test_subword = preprocess_corpus_with_subword(test_sents, subword_tokenizer)

    # Construir vocabulario
    vocab_subword, idx_to_word_subword = build_vocabulary(train_subword, min_freq=2)
    print(f"Tamaño vocabulario subword: {len(vocab_subword)}")

    # Crear datos
    X_train_sub, y_train_sub = create_ngram_data(train_subword, vocab_subword)
    X_test_sub, y_test_sub = create_ngram_data(test_subword, vocab_subword)

    # DataLoaders
    train_dataset_sub = list(zip(X_train_sub, y_train_sub))
    test_dataset_sub = list(zip(X_test_sub, y_test_sub))

    train_loader_sub = DataLoader(train_dataset_sub, batch_size=BATCH_SIZE, shuffle=True)
    test_loader_sub = DataLoader(test_dataset_sub, batch_size=BATCH_SIZE)

    # Entrenar modelo subword
    model_subword = TrigramModel(len(vocab_subword), EMBEDDING_DIM, CONTEXT_SIZE, H).to(device)
    model_subword = train_model(model_subword, train_loader_sub, len(vocab_subword), device)

    # Evaluar perplexidad
    perplexity_subword = calculate_perplexity(model_subword, test_loader_sub, device)
    print(f"Perplexidad modelo subword: {perplexity_subword:.2f}")

    # Calcular OOV rate subword
    all_test_subwords = set()
    for sent in test_subword:
        all_test_subwords.update(sent)
    vocab_sub_set = set(vocab_subword.keys())
    oov_subwords = all_test_subwords - vocab_sub_set
    oov_rate_subword = len(oov_subwords) / len(all_test_subwords) if all_test_subwords else 0
    print(f"OOV Rate subword: {oov_rate_subword:.2%}")

    # ==================== GENERACIÓN DE TEXTO ====================
    print("\n=== GENERACIÓN DE TEXTO ===")

    print("\nModelo Word-Based:")
    for i in range(3):
        generated = generate_text(model_word, vocab_word, idx_to_word_word)
        print(f"  Ejemplo {i+1}: {generated}")

    print("\nModelo Subword:")
    for i in range(3):
        generated = generate_text(model_subword, vocab_subword, idx_to_word_subword, subword_tokenizer)
        print(f"  Ejemplo {i+1}: {generated[:100]}...")

    # ==================== TABLA COMPARATIVA ====================
    print("\n=== ANÁLISIS COMPARATIVO ===")
    print("-" * 60)
    print(f"| {'Métrica':<20} | {'Modelo Base':>15} | {'Modelo Subword':>15} |")
    print("-" * 60)
    print(f"| {'Perplexidad (genesis)':<20} | {perplexity_word:>15.2f} | {perplexity_subword:>15.2f} |")
    print(f"| {'Tamaño vocabulario':<20} | {len(vocab_word):>15} | {len(vocab_subword):>15} |")
    print(f"| {'OOV Rate':<20} | {oov_rate_word:>14.2%} | {oov_rate_subword:>14.2%} |")
    print("-" * 60)


    from google.colab import drive
    drive.mount('/content/drive')

    # Guardar en Google Drive
    torch.save(model_word.state_dict(), '/content/drive/MyDrive/model_word_based.pt')
    torch.save(model_subword.state_dict(), '/content/drive/MyDrive/model_subword.pt')
    print("\nModelos guardados como 'model_word_based.pt' y 'model_subword.pt'")

    return {
        'word_based': {'perplexity': perplexity_word, 'vocab_size': len(vocab_word), 'oov_rate': oov_rate_word},
        'subword': {'perplexity': perplexity_subword, 'vocab_size': len(vocab_subword), 'oov_rate': oov_rate_subword}
    }

if __name__ == "__main__":
    results = main()

Usando dispositivo: cuda

=== CARGANDO CORPORA ===
  - gutenberg: 98552 oraciones
  - inaugural: 5395 oraciones
  - state_union: 17930 oraciones
  - webtext: 25733 oraciones
  - abc: 29059 oraciones
  - genesis (test): 13640 oraciones

=== ENTRENANDO MODELO BASE (Word-Based) ===
Tamaño vocabulario word-based: 32631
Epoch 1/3, Loss: 5.5269
Epoch 2/3, Loss: 5.1752
Epoch 3/3, Loss: 5.0080
Perplexidad modelo word-based: 127.25
OOV Rate word-based: 84.43%

=== ENTRENANDO MODELO SUBWORD ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1558 > 1024). Running this sequence through the model will result in indexing errors


Tamaño vocabulario subword: 34240
Epoch 1/3, Loss: 5.7104
Epoch 2/3, Loss: 5.2411
Epoch 3/3, Loss: 5.0387
Perplexidad modelo subword: 3772.37
OOV Rate subword: 10.55%

=== GENERACIÓN DE TEXTO ===

Modelo Word-Based:
  Ejemplo 1: a short the <UNK> <UNK> <UNK> of the lord ? <EOS>
  Ejemplo 2: he had the curious way . <EOS>
  Ejemplo 3: is there ; and as <UNK> nor all , and on the table , he saw you , and to <UNK> , and with the denying the ammonites , and

Modelo Subword:
  Ejemplo 1: ĠmoreĠconsiderĠinĠthisĠsubstanceĠ,ĠputĠforthĠtheirĠheartĠ,ĠthatĠtheyĠwouldĠnotĠmeetĠifĠyouĠwereĠ,Ġth...
  Ejemplo 2: ĠIĠwantĠtoĠgetĠonĠinĠtheĠworldĠ....
  Ejemplo 3: Ġ...ĠIĠseeĠhisĠfatherĠ,ĠsheĠhadĠnotĠbeenĠgivenĠtoĠtheĠdinnerĠwasĠtheĠgateĠofĠtheĠLORDĠyourĠGodĠ'ĠsĠh...

=== ANÁLISIS COMPARATIVO ===
------------------------------------------------------------
| Métrica              |     Modelo Base |  Modelo Subword |
------------------------------------------------------------
| Perplexidad (genesis) |    